# Exportar a GGUF

## Convierte el modelo fine-tuned a formato GGUF para inference local

GGUF permite ejecutar el modelo con llama.cpp en CPU o GPU.

In [ ]:
# @title 1. Instalar dependencias
!pip install -q llama-cpp-python scikit-build
!pip install -q ctransformers  # Alternativa mas rapida

In [ ]:
# @title 2. Montar Google Drive (opcional)
from google.colab import drive
drive.mount('/content/drive')

MODEL_PATH = "/content/drive/MyDrive/parasyte_cdp_expert"
OUTPUT_PATH = "/content/drive/MyDrive/parasyte_cdp_expert.gguf"

In [ ]:
# @title 3. Convertir a GGUF con llama.cpp
# Opcion A: Usar huggingface_hub (recomendado)
!pip install -q huggingface_hub

# Descargar convertidor de llama.cpp
!git clone https://github.com/ggerganov/llama.cpp.git
!cd llama.cpp && git pull origin master 2>/dev/null || true

In [ ]:
# @title 4. Asegurarse de que el modelo este en formato safetensors
import os

if os.path.exists(MODEL_PATH):
    print(f"Modelo encontrado en: {MODEL_PATH}")
    print("Archivos:")
    for f in os.listdir(MODEL_PATH):
        print(f"  - {f}")
else:
    print("ERROR: Modelo no encontrado")
    print("Primero ejecuta el notebook de entrenamiento")

In [ ]:
# @title 5. Convertir modelo
# Este paso puede tomar tiempo dependiendo del tamano del modelo

# Opcion: Usar convert.py de llama.cpp
# !cd llama.cpp && python convert.py $MODEL_PATH --outfile $OUTPUT_PATH --outtype f16

# O usar cuantizacion directa
# !cd llama.cpp && python convert.py $MODEL_PATH --outfile /tmp/model_f16.gguf --outtype f16
# !./quantize /tmp/model_f16.gguf $OUTPUT_PATH Q4_K_M

print("""
INSTRUCCIONES PARA CONVERTIR:

1. Descarga el modelo fine-tuned (directorio completo)
2. Localmente, ejecuta:
   git clone https://github.com/ggerganov/llama.cpp
   cd llama.cpp
   mkdir build && cd build
   cmake .. && make
   cd ..
   python convert.py /path/to/model --outfile model_f16.gguf --outtype f16
   ./quantize model_f16.gguf model_q4_k_m.gguf Q4_K_M

3. O usa el script automate_export.sh del repo
""")

In [ ]:
# @title 6. Inference de prueba con ctransformers
#!pip install -q ctransformers

# Solo si tienes el modelo en formato GGUF localmente:
#
# from ctransformers import AutoModelForCausalLM
#
# model = AutoModelForCausalLM.from_pretrained(
#     "path/to/model.gguf",
#     model_type="llama",
#     gpu_layers=32  # Usar GPU si esta disponible
# )
#
# prompt = "Eres un experto en Chrome DevTools Protocol. Instruction: Cuanta memoria usa JavaScript? CDP Command:"
# print(model(prompt, max_new_tokens=64))

---
## Metodos alternativos

### 1. Usar ollama
```bash
ollama create parasyte -f Modelfile
# Donde Modelfile contiene:
# FROM model.gguf
# PARAMETER temperature 0.1
# TEMPLATE """{{ .System }}
Eres un experto en CDP.
Instruction: {{ .Prompt }}
CDP Command:"""
```

### 2. Usar text-generation-webui
1. Descarga el modelo GGUF
2. Carga en la pestana "Model"
3. Selecciona quantization (Q4_K_M recomendado)

### 3. API local con LocalAI
```yaml
# docker-compose.yml
version: '1.0'
services:
  localai:
    image: quay.io/go-skynet/local-ai:latest
    volumes:
      - ./models:/models
    environment:
      - MODELS_PATH=/models
```

## Tamaños de quantized

| Quantization | Tamano | Calidad |
|--------------|--------|----------|
| Q2_K | ~0.8GB | Muy alta |
| Q3_K_M | ~1.0GB | Alta |
| Q4_K_M | ~1.2GB | Muy alta (recomendado) |
| Q5_K_M | ~1.4GB | Excelente |
| Q8_0 | ~2.4GB | Casi sin perdida |